# 📊 02 – Exploratory Data Analysis (EDA)
**Proyek:** AI-Based Pharmaceutical Data Selection & Monitoring System  
**Tim:** PJK-GM016 | Pijak × IBM SkillsBuild  
**Minggu:** 2 – EDA

> ⚠️ **Jalankan `01_preprocessing.ipynb` terlebih dahulu** agar file `rekap_produksi_clean.csv` tersedia.

---
### Visualisasi yang dihasilkan
| # | Grafik |
|---|--------|
| 1 | Distribusi label Defect Overall |
| 2 | Distribusi batch per produk |
| 3 | Distribusi variabel numerik kunci |
| 4 | Boxplot Normal vs Defect |
| 5 | Heatmap korelasi |
| 6 | Tren % Yield teoritis |
| 7 | Defect rate per material |
| 8 | Scatter plot Cetak vs Kemas |
| 9 | Ringkasan insight & rekomendasi |


## ⚙️ Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.15)
COLORS = {"normal": "#2ecc71", "defect": "#e74c3c"}

INPUT_CSV = "rekap_produksi_clean.csv"
df = pd.read_csv(INPUT_CSV, low_memory=False)

print(f"✅ Data dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head(3)

## 1️⃣ Distribusi Label Defect Overall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Distribusi Defect Overall", fontsize=15, fontweight='bold', y=1.02)

label_map = {0: 'Normal', 1: 'Defect'}
counts = df['Defect_Overall'].value_counts().rename(index=label_map)

# Pie
axes[0].pie(
    counts,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=[COLORS['normal'], COLORS['defect']],
    startangle=140,
    explode=(0.04, 0.04),
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title("Proporsi Normal vs Defect")

# Bar
bars = axes[1].bar(counts.index, counts.values,
                   color=[COLORS['normal'], COLORS['defect']],
                   edgecolor='white', width=0.4)
axes[1].set_title("Jumlah Baris per Kelas")
axes[1].set_ylabel("Jumlah Batch")
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(int(bar.get_height())), ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print("\n📌 Catatan: Dataset ini imbalanced.")
print(f"   Defect rate: {counts.get('Defect',0)/counts.sum()*100:.1f}%")
print("   → Gunakan class_weight='balanced' atau SMOTE saat modeling.")

## 2️⃣ Distribusi Batch per Produk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Distribusi Batch per Produk", fontsize=14, fontweight='bold')

mat_counts = df['Material_Description'].value_counts()
palette    = sns.color_palette("Set2", len(mat_counts))

# Total batch
axes[0].barh(mat_counts.index, mat_counts.values, color=palette, edgecolor='white')
axes[0].set_xlabel("Jumlah Batch")
axes[0].set_title("Total Batch")
for i, v in enumerate(mat_counts.values):
    axes[0].text(v + 1, i, str(v), va='center', fontsize=9)

# Defect per produk
defect_by_mat = df.groupby('Material_Description')['Defect_Overall'].sum().reindex(mat_counts.index)
axes[1].barh(defect_by_mat.index, defect_by_mat.values, color='#e74c3c', edgecolor='white', alpha=0.8)
axes[1].set_xlabel("Jumlah Batch Defect")
axes[1].set_title("Batch Defect per Produk")
for i, v in enumerate(defect_by_mat.values):
    axes[1].text(v + 0.1, i, str(int(v)), va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3️⃣ Distribusi Variabel Numerik Kunci

In [ ]:
numeric_keys = [
    'GB_Yield_Total', 'GK_Yield_Total',
    'Cetak_Yield_Kg', 'Cetak_Pct_Teoritis',
    'Kemas_Pct_Teoritis', 'Total_Waste_Kg',
    'Rasio_GK_GB', 'GB_Kadar_Air_Mean'
]
numeric_keys = [c for c in numeric_keys if c in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
fig.suptitle("Distribusi Variabel Numerik Utama", fontsize=14, fontweight='bold')

for i, col in enumerate(numeric_keys):
    data_c = df[col].dropna()
    axes[i].hist(data_c, bins=30, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i].axvline(data_c.mean(),   color='red',    linestyle='--', lw=1.5, label=f'Mean={data_c.mean():.2f}')
    axes[i].axvline(data_c.median(), color='orange', linestyle='--', lw=1.5, label=f'Median={data_c.median():.2f}')
    axes[i].set_title(col.replace('_', ' '), fontsize=9)
    axes[i].legend(fontsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 4️⃣ Boxplot: Normal vs Defect

In [ ]:
compare_cols = [
    'Cetak_Pct_Teoritis', 'Kemas_Pct_Teoritis',
    'GB_Yield_Total', 'Total_Waste_Kg', 'GB_Kadar_Air_Mean'
]
compare_cols = [c for c in compare_cols if c in df.columns]

df_plot = df.copy()
df_plot['Kelas'] = df_plot['Defect_Overall'].map({0: 'Normal', 1: 'Defect'})

fig, axes = plt.subplots(1, len(compare_cols), figsize=(18, 6))
fig.suptitle("Perbandingan Distribusi: Normal vs Defect", fontsize=14, fontweight='bold')

for i, col in enumerate(compare_cols):
    sns.boxplot(
        data=df_plot, x='Kelas', y=col,
        palette=[COLORS['normal'], COLORS['defect']],
        ax=axes[i], width=0.5,
        order=['Normal', 'Defect']
    )
    # Tambah titik rata-rata
    means = df_plot.groupby('Kelas')[col].mean()
    for j, (kelas, mean_val) in enumerate(means.items()):
        axes[i].plot(j, mean_val, 'D', color='white', markersize=7,
                     markeredgecolor='black', zorder=5)
    axes[i].set_title(col.replace('_', ' '), fontsize=10)
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

print("💡 Titik berlian (◆) = nilai rata-rata per kelas")

## 5️⃣ Heatmap Korelasi

In [ ]:
corr_cols = [c for c in numeric_keys + ['Defect_Overall', 'Defect_Cetak', 'Defect_Kemas']
             if c in df.columns]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 9},
    square=True
)
ax.set_title("Heatmap Korelasi Antar Variabel", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🔍 Korelasi tertinggi dengan Defect_Overall:")
print(corr_matrix['Defect_Overall'].drop('Defect_Overall').sort_values(
    key=abs, ascending=False).head(8).round(4).to_string())

## 6️⃣ Tren % Yield Terhadap Teoritis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=False)
fig.suptitle("Tren % Yield Terhadap Teoritis per Batch", fontsize=14, fontweight='bold')

for ax, col, title, threshold in zip(
    axes,
    ['Cetak_Pct_Teoritis', 'Kemas_Pct_Teoritis'],
    ['Cetak (% Teoritis)', 'Kemas/Primer (% Teoritis)'],
    [0.90, 0.90]
):
    data_s = df[col].dropna().reset_index(drop=True)
    ax.plot(data_s.index, data_s.values, color='#3498db', linewidth=0.7, alpha=0.8, label='% Teoritis')
    ax.axhline(threshold, color='red', linestyle='--', linewidth=1.5, label=f'Batas defect ({threshold:.0%})')
    ax.axhline(data_s.mean(), color='orange', linestyle='--', linewidth=1.2,
               label=f'Mean ({data_s.mean():.3f})')
    ax.fill_between(data_s.index, data_s.values, threshold,
                    where=(data_s.values < threshold),
                    color='red', alpha=0.15, label='Zona defect')
    # Tandai titik defect
    defect_idx = data_s[data_s < threshold].index
    ax.scatter(defect_idx, data_s[defect_idx], color='red', s=20, zorder=5, label=f'Defect ({len(defect_idx)})')
    ax.set_ylabel(title, fontsize=10)
    ax.set_xlabel("Index Batch", fontsize=9)
    ax.legend(fontsize=9, loc='upper right')
    ax.set_ylim([max(0, data_s.min() * 0.93), data_s.max() * 1.03])

plt.tight_layout()
plt.show()

## 7️⃣ Defect Rate per Material

In [ ]:
defect_rate = (
    df.groupby('Material_Description')['Defect_Overall']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'Defect', 'count': 'Total'})
)
defect_rate['Defect_Rate_%'] = (defect_rate['Defect'] / defect_rate['Total'] * 100).round(2)
defect_rate = defect_rate.sort_values('Defect_Rate_%', ascending=False)

fig, ax = plt.subplots(figsize=(13, 6))
colors_bar = ['#e74c3c' if r > 20 else '#f39c12' if r > 10 else '#2ecc71'
              for r in defect_rate['Defect_Rate_%']]
bars = ax.barh(defect_rate.index, defect_rate['Defect_Rate_%'],
               color=colors_bar, edgecolor='white', height=0.6)

ax.axvline(20, color='#e74c3c', linestyle='--', linewidth=1.2, label='Tinggi (>20%)')
ax.axvline(10, color='#f39c12', linestyle='--', linewidth=1.2, label='Sedang (>10%)')
ax.set_xlabel("Defect Rate (%)", fontsize=11)
ax.set_title("Defect Rate per Produk", fontweight='bold', fontsize=13)
for bar in bars:
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontsize=10, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print("\n📋 Tabel Defect Rate:")
print(defect_rate[['Defect', 'Total', 'Defect_Rate_%']].to_string())

## 8️⃣ Scatter: Cetak vs Kemas % Teoritis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Analisis % Teoritis: Cetak vs Kemas", fontsize=14, fontweight='bold')

scatter_df = df[['Cetak_Pct_Teoritis', 'Kemas_Pct_Teoritis',
                  'Defect_Overall', 'Material_Description']].dropna()
scatter_df['Kelas'] = scatter_df['Defect_Overall'].map({0: 'Normal', 1: 'Defect'})

# Scatter plot
for kelas, color in [('Normal', COLORS['normal']), ('Defect', COLORS['defect'])]:
    mask = scatter_df['Kelas'] == kelas
    axes[0].scatter(
        scatter_df.loc[mask, 'Cetak_Pct_Teoritis'],
        scatter_df.loc[mask, 'Kemas_Pct_Teoritis'],
        label=kelas, color=color, alpha=0.5, s=30, edgecolors='none'
    )
axes[0].axvline(0.90, color='gray', linestyle='--', linewidth=1, alpha=0.7)
axes[0].axhline(0.90, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Threshold 90%')
axes[0].set_xlabel("Cetak % Teoritis")
axes[0].set_ylabel("Kemas % Teoritis")
axes[0].set_title("Scatter: Cetak vs Kemas")
axes[0].legend()

# Violin plot
df_plot = df[['Kemas_Pct_Teoritis', 'Defect_Overall']].dropna().copy()
df_plot['Kelas'] = df_plot['Defect_Overall'].map({0: 'Normal', 1: 'Defect'})
sns.violinplot(data=df_plot, x='Kelas', y='Kemas_Pct_Teoritis',
               palette=[COLORS['normal'], COLORS['defect']], ax=axes[1],
               order=['Normal', 'Defect'], inner='box')
axes[1].axhline(0.90, color='red', linestyle='--', linewidth=1.5, label='Threshold 90%')
axes[1].set_title("Violin: Kemas % Teoritis per Kelas")
axes[1].set_xlabel('')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9️⃣ Ringkasan Insight & Rekomendasi

In [ ]:
print("=" * 65)
print("  RINGKASAN INSIGHT EDA – PJK-GM016")
print("=" * 65)

total       = len(df)
n_defect    = df['Defect_Overall'].sum()
n_normal    = total - n_defect
defect_rate = n_defect / total * 100

print(f"\n📦 Total batch dianalisis : {total}")
print(f"   Normal                 : {n_normal} ({100-defect_rate:.1f}%)")
print(f"   Defect                 : {n_defect} ({defect_rate:.1f}%)")

print(f"\n🏭 Jumlah produk unik     : {df['Material_Description'].nunique()}")

top_defect = defect_rate_sorted = (
    df.groupby('Material_Description')['Defect_Overall']
    .agg(['sum','count'])
    .assign(rate=lambda x: x['sum']/x['count']*100)
    .sort_values('rate', ascending=False)
    .head(3)
)
print(f"\n⚠️  Produk defect tertinggi:")
for mat, row in top_defect.iterrows():
    print(f"   {mat[:40]:40s} → {row['rate']:.1f}% ({int(row['sum'])}/{int(row['count'])})")

if 'Cetak_Pct_Teoritis' in df.columns:
    corr_val = df[['Cetak_Pct_Teoritis','Kemas_Pct_Teoritis','Defect_Overall']].corr()['Defect_Overall']
    print(f"\n📈 Korelasi dengan Defect_Overall:")
    print(f"   Cetak_Pct_Teoritis  : {corr_val.get('Cetak_Pct_Teoritis', 'N/A'):.4f}")
    print(f"   Kemas_Pct_Teoritis  : {corr_val.get('Kemas_Pct_Teoritis', 'N/A'):.4f}")

print(f"\n💡 Rekomendasi untuk Modeling (Minggu 3):")
print(f"   1. Gunakan class_weight='balanced' pada Random Forest")
print(f"      karena data imbalanced ({defect_rate:.1f}% defect)")
print(f"   2. Fokus pada fitur: Cetak_Pct_Teoritis, Kemas_Pct_Teoritis,")
print(f"      GB_Yield_Total, Total_Waste_Kg, GB_Kadar_Air_Mean")
print(f"   3. Evaluation metric utama: Recall & F1-Score (bukan Accuracy)")
print(f"   4. Lakukan StratifiedKFold CV (k=5) untuk validasi robust")
print("=" * 65)
print("\n✅ EDA selesai! Lanjut ke 03_modeling.ipynb (Minggu 3)")